#### Imports

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import pickle
import warnings
import os
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')

from kmrf import KMRF
from KMRF_training_config import *
from SIMULATOR import SIMULATOR
from OPTIMIZER_INPUTS import OPTIMIZER_INPUTS


# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


## Multi-Horizon Regime Predictions

#### Global Vars and Helper Functions

In [18]:
KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')
KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')

# Using saved KAMA+MSR models
def get_KM_model_dates(KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) -> pd.Series:
    return pd.Series([f.stem for f in list(KM_MODEL_BASE_PATH.glob('*'))]).sort_values().iloc[1:].reset_index(drop=True)

def get_KM_model_paths(MODEL_DATE: str, KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) ->  pd.Series:
    return pd.Series(list((KM_MODEL_BASE_PATH / MODEL_DATE).glob('*'))).sort_values().reset_index(drop=True)

# Using saved KMRF predictions
def get_asset_names(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    kmrf_preds_paths = list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))
    return pd.Series([f.stem.split('multi')[0][:-1].replace('_', ' ') for f in kmrf_preds_paths]).sort_values().reset_index(drop=True)

def get_KMRF_prediction_paths(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    return pd.Series(list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))).sort_values().reset_index(drop=True)

In [19]:
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

TRADING_DAYS = CustomBusinessDay(calendar=USFederalHolidayCalendar())

#### --------------------------------------------------------

In [20]:
RISK_FREE_RATES = pd.read_csv('data/risk_free_rates.csv', parse_dates=['Date'], index_col='Date')
RISK_FREE_RATES = RISK_FREE_RATES['RF_3M'].rename('RF').to_frame()

In [21]:
KM_MODEL_DATES = get_KM_model_dates()
ASSET_NAMES = get_asset_names()

display(ASSET_NAMES)

0     Consumer Discretionary Select Sector SPDR
1           Consumer Staples Select Sector SPDR
2                     Energy Select Sector SPDR
3                  Financial Select Sector SPDR
4                Health Care Select Sector SPDR
5                 Industrial Select Sector SPDR
6                        Invesco DB Agriculture
7                             Invesco QQQ Trust
8                  Materials Select Sector SPDR
9         SPDR Dow Jones Industrial Average ETF
10                             SPDR Gold Shares
11                             SPDR S&P 500 ETF
12                Technology Select Sector SPDR
13               United States Natural Gas Fund
14                       United States Oil Fund
15                 Utilities Select Sector SPDR
16          Vanguard FTSE Developed Markets ETF
17           Vanguard FTSE Emerging Markets ETF
18                     Vanguard FTSE Europe ETF
19                  iShares China Large-Cap ETF
20                       iShares MSCI In

In [22]:
KM_MODEL_DATES.iloc[-12]

'20241101'

In [23]:
PORTFOLIO_ASSETS = ['Technology Select Sector SPDR', 'Health Care Select Sector SPDR', 'Financial Select Sector SPDR',
                    'Consumer Discretionary Select Sector SPDR', 'Consumer Staples Select Sector SPDR', 
                    'Industrial Select Sector SPDR', 'Energy Select Sector SPDR', 'Utilities Select Sector SPDR',
                    'Materials Select Sector SPDR', 'iShares U.S. Real Estate ETF']
# Add a day first because optimization dates are the first day of the prediction period
# even though we want to optimize at the end of the previous trading day
# e.g., optimize on 2018-12-31 for the 21 trading day period starting 2019-01-02
# Risk-free rate is taken from the previous trading day
OPT_DATE = pd.Timestamp(KM_MODEL_DATES.iloc[-12]) + TRADING_DAYS
RISK_FREE = RISK_FREE_RATES.loc[OPT_DATE - TRADING_DAYS].values[0]
OPT_INPUTS = OPTIMIZER_INPUTS(opt_date=OPT_DATE, asset_list=PORTFOLIO_ASSETS, risk_free_rate=RISK_FREE, 
                              n_days=21, n_simulations=250, random_seed=42)
print("Optimization Date:", OPT_INPUTS.opt_date.date())
print(f"Risk-Free Rate ({(OPT_DATE - TRADING_DAYS).date()}):", OPT_INPUTS.risk_free_rate)

Optimization Date: 2024-11-04
Risk-Free Rate (2024-11-01): 0.0461


In [24]:
OPT_INPUTS.run_full_pipeline()


################################################################################
# OPTIMIZER INPUTS FULL PIPELINE
# Optimization Date: 2024-11-04
# Assets: 10
# Simulations: 1,000
# Horizon: 21 days
################################################################################

LOADING SIMULATORS FOR 10 ASSETS
Optimization Date: 2024-11-04
Model Date: 20241101
  [1/10] Loading Technology Select Sector SPDR...
  [2/10] Loading Health Care Select Sector SPDR...
  [3/10] Loading Financial Select Sector SPDR...
  [4/10] Loading Consumer Discretionary Select Sector SPDR...
  [5/10] Loading Consumer Staples Select Sector SPDR...
  [6/10] Loading Industrial Select Sector SPDR...
  [7/10] Loading Energy Select Sector SPDR...
  [8/10] Loading Utilities Select Sector SPDR...
  [9/10] Loading Materials Select Sector SPDR...
  [10/10] Loading iShares U.S. Real Estate ETF...

✓ Successfully loaded 10/10 simulators

⚠️  Market asset 'SPDR S&P 500 ETF' not in portfolio. Loading separately...

ESTI

In [25]:
OPT_INPUTS.optimize_portfolio(objective='max_sharpe', allow_short=True, gross_exposure=2)


PORTFOLIO OPTIMIZATION
  Objective: max_sharpe
  Assets: 10
  Short selling: Allowed
  Gross exposure limit: 200.0%

  Optimization Results:
    Portfolio Return: 18.93%
    Portfolio Risk: 17.15%
    Sharpe Ratio: 0.835
    Sortino Ratio: 0.000

  Portfolio Weights:
    Consumer Discretionary Select Sector SPDR:   75.42%
    Technology Select Sector SPDR:   68.60%
    iShares U.S. Real Estate ETF:    5.98%
    Materials Select Sector SPDR:  -22.78%
    Health Care Select Sector SPDR:  -27.22%

✓ Portfolio optimized


In [26]:
OPT_INPUTS.portfolio_summary().round(4)

,Weight,Expected Return,Return Contribution,Marginal Risk,Risk Contribution,Risk Contribution %
Consumer Discretionary Select Sector SPDR,0.7542,0.1345,0.1015,0.1307,0.0986,57.4660
Technology Select Sector SPDR,0.6860,0.1493,0.1024,0.1483,0.1018,59.3302
Health Care Select Sector SPDR,-0.2722,0.0352,-0.0096,0.0611,-0.0166,-9.6947
Materials Select Sector SPDR,-0.2278,0.0437,-0.0100,0.0714,-0.0163,-9.4792
iShares U.S. Real Estate ETF,0.0598,0.0824,0.0049,0.0682,0.0041,2.3777
Utilities Select Sector SPDR,-0.0000,0.0177,-0.0000,0.0354,-0.0000,-0.0000
Industrial Select Sector SPDR,-0.0000,0.0680,-0.0000,0.0905,-0.0000,-0.0000
Financial Select Sector SPDR,-0.0000,0.0705,-0.0000,0.0862,-0.0000,-0.0000
Energy Select Sector SPDR,-0.0000,0.0601,-0.0000,0.0600,-0.0000,-0.0000
Consumer Staples Select Sector SPDR,-0.0000,0.0509,-0.0000,0.0465,-0.0000,-0.0000
